# Paso 1:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import random
import os

from torchvision import transforms
from sklearn.preprocessing import normalize
from PIL import Image
from tqdm import tqdm
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


# Paso 2

In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
    print("Ejecutando en Google Colab")
    base_path = Path('/content')
except ImportError:
    IN_COLAB = False
    print("Ejecutando localmente")
    base_path = Path(os.getcwd())

# Descargar según el entorno
if IN_COLAB:
    !gdown --id 11-TD6add7zZaukIB8l2cVnBTJgsTjo8n
else:
    # Para entorno local
    import gdown
    gdown.download(id='11-TD6add7zZaukIB8l2cVnBTJgsTjo8n', output='dataset_ecom_mini.zip')

# Descomprimir
!unzip -q -o dataset_ecom_mini.zip

# Cargar datos
data_dir = base_path / 'eval'
df = pd.read_csv(base_path / 'eval.csv', delimiter=';')

print(f"Dataset listo: {len(df)} imágenes en {data_dir}")
print(df.head())

Ejecutando en Google Colab
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=11-TD6add7zZaukIB8l2cVnBTJgsTjo8n
To: /content/dataset_ecom_mini.zip
100% 10.6M/10.6M [00:00<00:00, 42.6MB/s]
Dataset listo: 300 imágenes en /content/eval
    Title GlobalCategoryEN
0  14toys        Toy Store
1  16toys        Toy Store
2  24toys        Toy Store
3  60toys        Toy Store
4  84toys        Toy Store


# Paso 3:

In [3]:
def transform(image):
    transform_pipeline = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transform_pipeline(image)

# Codificar todas las imágenes
embeddings = []
valid_filenames = []

# Get all image filenames from the dataframe
image_filenames = df['Title'].values
data_dir = Path('eval') # Define data_dir here as well

for filename in tqdm(image_filenames):
    img_path = data_dir / f"{filename}.jpg"
    if img_path.exists():
        try:
            image = Image.open(img_path)
            transformed_image = transform(image)
            embeddings.append(transformed_image)
        except Exception as e:
            print(f"Error codificando {filename}: {e}")

# Convertir a array de numpy
print(f"\nCodificadas exitosamente {len(embeddings)} imágenes")

100%|██████████| 300/300 [00:03<00:00, 87.70it/s]


Codificadas exitosamente 300 imágenes


# Paso 4

In [5]:
# Cargar el modelo DinoV2 desde torch hub
# https://github.com/facebookresearch/dinov2

#model_v2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
model_v2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model_v2 = model_v2.to(device)
model_v2.eval()

print("¡Modelo DinoV2 cargado exitosamente!")
print(f"Dimensión de salida del modelo: {model_v2.embed_dim}")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 270MB/s]


¡Modelo DinoV2 cargado exitosamente!
Dimensión de salida del modelo: 384


In [12]:
import sys
sys.path.insert(0, './dinov3-main')

!gdown --id 1-LqJ2bS_T8XOx5TClDq31MCkPPoXO4bZ
!unzip dinov3-main.zip

model_v3 = torch.hub.load(repo_or_dir='./dinov3-main', model='dinov3_vitl16', source='local', weights='dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth')
model_v3 = model_v3.to(device)
model_v3.eval()

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1-LqJ2bS_T8XOx5TClDq31MCkPPoXO4bZ
To: /content/dinov3-main.zip
100% 10.3M/10.3M [00:00<00:00, 252MB/s]
Archive:  dinov3-main.zip
adc254450203739c8149213a7a69d8d905b4fcfa
replace dinov3-main/.docstr.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: dinov3-main/.docstr.yaml  
  inflating: dinov3-main/.github/workflows/lint.yaml  
  inflating: dinov3-main/.gitignore  
  inflating: dinov3-main/CODE_OF_CONDUCT.md  
  inflating: dinov3-main/CONTRIBUTING.md  
  inflating: dinov3-main/LICENSE.md  
  inflating: dinov3-main/MODEL_CARD.md  
  inflating: dinov3-main/README.md   
  inflating: dinov3-main/conda.yaml  
  inflating: dinov3-main/dinov3/__init__.py  
  inflating: dinov3-main/dinov3/checkpointer/__init__.py  
 

URLError: <urlopen error [Errno 2] No such file or directory: '/content/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'>

In [ ]:
# Paso 5: Codificar imágenes usando el modelo preentrenad

In [ ]:
def encode_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model(img_tensor)

    return features.cpu().numpy().flatten()


# Cargar el modelo preentrenado (por ejemplo, ResNet50)

# Obtener todos los nombres de archivos de imágenes del dataframe
image_filenames = df['Title'].values
n_images = len(image_filenames)

print(f"Codificando {n_images} imágenes...")

# Codificar todas las imágenes
embeddings = []
valid_filenames = []

for filename in tqdm(image_filenames):
    img_path = data_dir / f"{filename}.jpg"

    if img_path.exists():
        try:
            embedding = encode_image(img_path)
            embeddings.append(embedding)
            valid_filenames.append(filename)
        except Exception as e:
            print(f"Error codificando {filename}: {e}")

# Convertir a array de numpy
embeddings = np.array(embeddings)
print(f"\nCodificadas exitosamente {len(embeddings)} imágenes")
print(f"Forma del embedding: {embeddings.shape}")